# If run on Colab:

In [ ]:
!pip install transformers sentence-transformers scikit-learn pandas numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.6 MB/s eta 0:00:00


# Data Loading

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from sklearn import preprocessing
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import BertTokenizer, BertModel
import torch
import torch.nn as nn
import torch.optim as optim

# load_aokvqa.py
import os
import json

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def load_aokvqa(aokvqa_dir, split, version='v1p0'):
    assert split in ['train', 'val', 'test', 'test_w_ans']
    dataset = json.load(open(
        os.path.join(aokvqa_dir, f"aokvqa_{version}_{split}.json")
    ))
    return dataset

def get_coco_path(split, image_id, coco_dir):
    return os.path.join(coco_dir, f"{split}2017", f"{image_id:012}.jpg")

USE_COLAB = False
if USE_COLAB:
    # Colab File Paths
    # AOKVQA_DIR="/content"
    # COCO_DIR="datasets/coco/"
    aokvqa_dir = "/content/"
    # coco_dir = f"/content/"
else:
    AOKVQA_DIR="datasets/aokvqa/"
    COCO_DIR="datasets/coco/"
    aokvqa_dir = f"./aokvqa/{AOKVQA_DIR}"
    coco_dir = f"./aokvqa/{COCO_DIR}"



Using device: cpu


In [2]:
def subsample(dataset, n_samples=None, frac=None, random_seed=42):
    random.seed(random_seed)
    if n_samples:
        return random.sample(dataset, n_samples)
    elif frac:
        sample_size = int(len(dataset) * frac)
        return random.sample(dataset, sample_size)
    return dataset


In [3]:
USE_SUBSET_DATA = False
train_dataset = load_aokvqa(aokvqa_dir, 'train')
val_dataset = load_aokvqa(aokvqa_dir, 'val')
test_dataset = load_aokvqa(aokvqa_dir, 'test')
print(f"Full Train aokvqa: {len(train_dataset)}")
print(f"Full Val aokvqa: {len(val_dataset)}")
print(f"Full Test aokvqa: {len(test_dataset)}")

if USE_SUBSET_DATA:
    train_dataset = subsample(train_dataset, frac=0.2)
    val_dataset = subsample(val_dataset, frac=0.2)
    test_dataset = subsample(test_dataset, frac=0.2)
    print(f"Used Train aokvqa: {len(train_dataset)}")
    print(f"Used Val aokvqa: {len(val_dataset)}")
    print(f"Used Test aokvqa: {len(test_dataset)}")





Full Train aokvqa: 17056
Full Val aokvqa: 1145
Full Test aokvqa: 6702


In [4]:
val_dataset[0]

{'split': 'val',
 'image_id': 461751,
 'question_id': '22jbM6gDxdaMaunuzgrsBB',
 'question': "What is in the motorcyclist's mouth?",
 'choices': ['toothpick', 'food', 'popsicle stick', 'cigarette'],
 'correct_choice_idx': 3,
 'direct_answers': ['cigarette',
  'cigarette',
  'cigarette',
  'cigarette',
  'cigarette',
  'cigarette',
  'cigarette',
  'cigarette',
  'cigarette',
  'cigarette'],
 'difficult_direct_answer': False,
 'rationales': ["He's smoking while riding.",
  'The motorcyclist has a lit cigarette in his mouth while he rides on the street.',
  'The man is smoking.']}

# Data Preparation

Fields Considered:
- Question
- Choices
- Correct answer
- Correct Choice Indice
- rationale
- direct_answer

In [5]:
# Convert train_dataset into train_df
train_data = []
for sample in train_dataset:
    question = sample["question"]
    choices = sample["choices"]
    correct_choice_idx = sample["correct_choice_idx"]
    rationales = sample.get("rationales", [])
    combined_rationale = " ".join(rationales)
    direct_answers = sample.get("direct_answers", [])
    combined_direct_answer = " ".join(direct_answers)
    correct_ans = choices[correct_choice_idx]

    train_data.append({
        "question": question,
        "choices": choices,
        "correct_answer": correct_ans,
        "correct_choice_idx": correct_choice_idx,
        "rationale": combined_rationale,
        "direct_answer": combined_direct_answer
    })

train_df = pd.DataFrame(train_data)

# Convert test_dataset into test_df
val_data = []
for sample in val_dataset:
    question = sample["question"]
    choices = sample["choices"]
    correct_choice_idx = sample["correct_choice_idx"]
    rationales = sample.get("rationales", [])
    combined_rationale = " ".join(rationales)
    direct_answers = sample.get("direct_answers", [])
    combined_direct_answer = " ".join(direct_answers)
    correct_ans = choices[correct_choice_idx]

    val_data.append({
        "question": question,
        "choices": choices,
        "correct_answer": correct_ans,
        "correct_choice_idx": correct_choice_idx,
        "rationale": combined_rationale,
        "direct_answer": combined_direct_answer
    })

val_df = pd.DataFrame(val_data)

print(f"Train Set Size: {len(train_df)}")
print(f"Val Set Size: {len(val_df)}")


Train Set Size: 17056
Val Set Size: 1145


In [6]:
train_df.head()

,question,choices,correct_answer,correct_choice_idx,rationale,direct_answer
0,What is the man by the bags awaiting?,"[skateboarder, train, delivery, cab]",cab,3,"A train would not be on the street, he would n...",ride ride bus taxi travelling traffic taxi cab...
1,Where does this man eat pizza?,"[office, cafe, motel, outside]",office,0,The man is eating pizza at a work desk in an o...,work office work work at work desk at desk off...
2,What is the occupation of the person driving?,"[waiter, farmer, cashier, musician]",farmer,1,The place is full of sheep that shows the pers...,farmer farmer bus driver farmer shepherd farme...
3,How were the drivers of the cars able to park ...,"[firemen, airport workers, police, postal work...",airport workers,1,These drivers work at the airport. Cars are pa...,airport workers driving stilts parking lot des...
4,How many people can ride this motorcycle at a ...,"[four, two, three, one]",two,1,Two people can be on the bike. There is a pass...,two two two two two two two two two two


# Unimodal Baselines (Text)

### Majority Class Baseline

In [7]:
qa_data = []
for sample in val_dataset:
    question_id = sample["question_id"]
    image_id = sample["image_id"]
    question = sample["question"]
    choices = sample["choices"]
    correct_choice_idx = sample["correct_choice_idx"]
    rationales = sample.get("rationales", [])
    combined_rationale = " ".join(rationales)
    direct_answers = sample.get("direct_answers", [])
    combined_direct_answer = " ".join(direct_answers)
    correct_ans = choices[correct_choice_idx]
    qa_data.append({
        "question_id": question_id,
        "image_id": image_id,
        "question": question,
        "choices": choices,
        "correct_answer": correct_ans,
        "correct_choice_idx": correct_choice_idx,
        "rationale": combined_rationale,
        "direct_answer": combined_direct_answer
    })
qa_df = pd.DataFrame(qa_data)

In [ ]:

from collections import Counter

# Count answer frequencies
all_answers = [sample['correct_answer'] for sample in qa_data]
answers_count = Counter(all_answers)

qa_df['majority_class_prediction'] = qa_df['choices'].apply(
    lambda choice_list: max(choice_list, key=lambda c: answers_count[c])
)
accuracy = (qa_df['majority_class_prediction'] == qa_df['correct_answer']).mean()
print(f"Majority Class Baseline Accuracy: {accuracy:.2%}")

y_true = (qa_df['correct_answer'] == qa_df['majority_class_prediction']).astype(int)
y_pred = np.ones_like(y_true)  # Baseline always predicts 1 (same class)



Majority Class Baseline Accuracy: 66.20%
Majority Class Baseline ROC AUC: 0.8163


In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelBinarizer

# Convert correct answers and predictions to numerical format
y_true = qa_df['correct_answer']
y_pred = qa_df['majority_class_prediction']

# Binarize the labels for multi-class ROC AUC
lb = LabelBinarizer()
y_true_bin = lb.fit_transform(y_true)
y_pred_bin = lb.transform(y_pred)  # The Majority Baseline always predicts the most frequent class

# Compute ROC AUC score
roc_auc = roc_auc_score(y_true_bin, y_pred_bin, average='macro')

print(f"Majority Class Baseline ROC AUC: {roc_auc:.4f}")


# Convert correct answers to numerical format
y_true = qa_df['correct_choice_idx'].values  # Correct choice indices

# Ensure majority prediction is mapped to an index within choices
def get_choice_index(row):
    try:
        return row['choices'].index(row['majority_class_prediction'])  # Get index of predicted choice
    except ValueError:
        return 0  # Default to index 0 if not found (fallback to first option)

qa_df['predicted_choice_idx'] = qa_df.apply(get_choice_index, axis=1)
y_pred_idx = qa_df['predicted_choice_idx'].values

# Convert to torch tensors
y_true_tensor = torch.tensor(y_true, dtype=torch.long)
y_pred_tensor = torch.tensor(y_pred_idx, dtype=torch.long)

# One-hot encoding for CrossEntropyLoss input
num_classes = max(qa_df['correct_choice_idx'].max(), qa_df['predicted_choice_idx'].max()) + 1
y_pred_probs = torch.zeros(len(y_pred_tensor), num_classes)  # Initialize probabilities
y_pred_probs[range(len(y_pred_tensor)), y_pred_idx] = 1.0  # Assign probability 1 to the predicted class

# Define Cross-Entropy Loss
criterion = nn.CrossEntropyLoss()
loss = criterion(y_pred_probs, y_true_tensor)

print(f"Majority Class Baseline Cross-Entropy Loss: {loss.item():.4f}")


Majority Class Baseline Cross-Entropy Loss: 1.0817


### TF-IDF + Logistic Regression (Question ONLY)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# **Step 1: Prepare Data - Create Pairs (Question + Choice)**
train_pairs = []
train_labels = []

val_pairs = []
val_labels = []
for _, row in train_df.iterrows():
    question = row["question"]
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice  # Combine Question + Choice
        train_pairs.append(text_input)
        train_labels.append(1 if idx == correct_idx else 0)  # Label: 1 if correct, 0 otherwise

for _, row in val_df.iterrows():
    question = row["question"]
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice
        val_pairs.append(text_input)
        val_labels.append(1 if idx == correct_idx else 0)

# **Step 2: Compute TF-IDF Embeddings**
vectorizer = TfidfVectorizer(max_features=10000)  # Use the 5000 most frequent words
X_train = vectorizer.fit_transform(train_pairs).toarray()
X_val = vectorizer.transform(val_pairs).toarray()

y_train = np.array(train_labels)
y_val = np.array(val_labels)

# **Step 3: Train Logistic Regression Model**
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

# **Step 4: Get Predictions (Probabilities for Each Choice)**
y_pred_probs = model_lr.predict_proba(X_val)[:, 1]  # Get probabilities for class 1 (correct choice)

# **Step 5: Compute Accuracy (Choice Selection)**
val_choice_indices = []  # Stores predicted choice index for each question
val_correct_indices = []  # Stores ground-truth indices

i = 0
for _, row in val_df.iterrows():
    num_choices = len(row["choices"])
    scores = y_pred_probs[i:i + num_choices]  # Get scores for this question's choices
    pred_choice_idx = np.argmax(scores)  # Pick highest-scoring choice
    val_choice_indices.append(pred_choice_idx)
    val_correct_indices.append(row["correct_choice_idx"])  # Ground-truth index
    i += num_choices

# **Final Accuracy Computation**
tfidf_lr_accuracy = accuracy_score(val_correct_indices, val_choice_indices)
print(f"TF-IDF + Logistic Regression Accuracy: {tfidf_lr_accuracy:.2%}")

# **Step 6: Print Sample Predictions**
print("\nSample Predictions:")
i = 0
for _, row in val_df.iterrows():
    if i >= 5:
        break
    print(f"Q: {row['question']}")
    print(f"Choices: {row['choices']}")
    print(f"Actual Answer: {row['choices'][row['correct_choice_idx']]}")
    print(f"Predicted Answer: {row['choices'][val_choice_indices[i]]}\n")
    i += 1
# **Cross-Entropy Loss (Log Loss)**
cross_entropy = log_loss(y_val, y_pred_probs)
print(f"Cross-Entropy Loss: {cross_entropy:.4f}")

# **Precision, Recall, and F1 Score**
# Since it's a binary classification problem (correct=1, incorrect=0), we compute these on y_val
y_pred_binary = (y_pred_probs >= 0.5).astype(int)  # Convert probabilities to 0/1 using a 0.5 threshold

precision = precision_score(y_val, y_pred_binary)
recall = recall_score(y_val, y_pred_binary)
f1 = f1_score(y_val, y_pred_binary)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, log_loss




Cross-Entropy Loss: 0.5450
Precision: 0.6512
Recall: 0.0245
F1-Score: 0.0471


### TF-IDF + Logistic Regression

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# **Step 1: Prepare Data - Create Pairs (Question + Choice)**
train_pairs = []
train_labels = []

val_pairs = []
val_labels = []

for _, row in train_df.iterrows():
    question = row["question"] + row['rationale']
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice  # Combine Question + Choice
        train_pairs.append(text_input)
        train_labels.append(1 if idx == correct_idx else 0)  # Label: 1 if correct, 0 otherwise

for _, row in val_df.iterrows():
    question = row["question"] + row['rationale']
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice
        val_pairs.append(text_input)
        val_labels.append(1 if idx == correct_idx else 0)

# **Step 2: Compute TF-IDF Embeddings**
vectorizer = TfidfVectorizer(max_features=10000)  # Use the 5000 most frequent words
X_train = vectorizer.fit_transform(train_pairs).toarray()
X_val = vectorizer.transform(val_pairs).toarray()

y_train = np.array(train_labels)
y_val = np.array(val_labels)

# **Step 3: Train Logistic Regression Model**
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

# **Step 4: Get Predictions (Probabilities for Each Choice)**
y_pred_probs = model_lr.predict_proba(X_val)[:, 1]  # Get probabilities for class 1 (correct choice)

# **Step 5: Compute Accuracy (Choice Selection)**
val_choice_indices = []  # Stores predicted choice index for each question
val_correct_indices = []  # Stores ground-truth indices

i = 0
for _, row in val_df.iterrows():
    num_choices = len(row["choices"])
    scores = y_pred_probs[i:i + num_choices]  # Get scores for this question's choices
    pred_choice_idx = np.argmax(scores)  # Pick highest-scoring choice
    val_choice_indices.append(pred_choice_idx)
    val_correct_indices.append(row["correct_choice_idx"])  # Ground-truth index
    i += num_choices

# **Final Accuracy Computation**
tfidf_lr_accuracy = accuracy_score(val_correct_indices, val_choice_indices)
print(f"TF-IDF + Logistic Regression Accuracy: {tfidf_lr_accuracy:.2%}")

# **Step 6: Print Sample Predictions**
print("\nSample Predictions:")
i = 0
for _, row in val_df.iterrows():
    if i >= 5:
        break
    print(f"Q: {row['question']}")
    print(f"Choices: {row['choices']}")
    print(f"Actual Answer: {row['choices'][row['correct_choice_idx']]}")
    print(f"Predicted Answer: {row['choices'][val_choice_indices[i]]}\n")
    i += 1
# **Cross-Entropy Loss (Log Loss)**
cross_entropy = log_loss(y_val, y_pred_probs)
print(f"Cross-Entropy Loss: {cross_entropy:.4f}")

# **Precision, Recall, and F1 Score**
# Since it's a binary classification problem (correct=1, incorrect=0), we compute these on y_val
from sklearn.metrics import precision_recall_curve

precision_vals, recall_vals, thresholds = precision_recall_curve(y_val, y_pred_probs)
best_threshold = thresholds[np.argmax(precision_vals * recall_vals)]  # F1-optimal threshold
y_pred_binary = (y_pred_probs >= best_threshold).astype(int)

precision = precision_score(y_val, y_pred_binary)
recall = recall_score(y_val, y_pred_binary)
f1 = f1_score(y_val, y_pred_binary)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

TF-IDF + Logistic Regression Accuracy: 63.84%

Sample Predictions:
Q: What is in the motorcyclist's mouth?
Choices: ['toothpick', 'food', 'popsicle stick', 'cigarette']
Actual Answer: cigarette
Predicted Answer: toothpick

Q: Which number birthday is probably being celebrated?
Choices: ['one', 'ten', 'nine', 'thirty']
Actual Answer: thirty
Predicted Answer: thirty

Q: What best describes the pool of water?
Choices: ['frozen', 'fresh', 'dirty', 'boiling']
Actual Answer: dirty
Predicted Answer: dirty

Q: What is the white substance on top of the cupcakes?
Choices: ['butter', 'mayo', 'ice cream', 'icing']
Actual Answer: icing
Predicted Answer: mayo

Q: What type of device is sitting next to the laptop?
Choices: ['mouse', 'mobile phone', 'pen', 'keyboard']
Actual Answer: mobile phone
Predicted Answer: mobile phone

Cross-Entropy Loss: 0.5541
Precision: 0.2514
Recall: 0.9983
F1-Score: 0.4016


In [ ]:
y_val, y_pred_probs

(array([0, 0, 0, ..., 0, 0, 0]),
 array([0.36977327, 0.36642324, 0.34633724, ..., 0.26996935, 0.26241127,
        0.24049053]))

## TFIDF + MLP (Question + Rationale)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import StepLR
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
import numpy as np

# **Step 1: Prepare Data - Create Pairs (Question + Choice)**
train_pairs = []
train_labels = []

val_pairs = []
val_labels = []

for i, row in train_df.iterrows():
    question = row["question"] + row['rationale']
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]  # This is the correct choice index

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice  # Combine Question + Choice
        train_pairs.append(text_input)
        train_labels.append(1 if idx == correct_idx else 0)  # Label: 1 if correct, 0 otherwise

for i, row in val_df.iterrows():
    question = row["question"] + row['rationale']
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice
        val_pairs.append(text_input)
        val_labels.append(1 if idx == correct_idx else 0)

# **Step 2: Compute TF-IDF Embeddings**
vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(train_pairs).toarray()
X_val = vectorizer.transform(val_pairs).toarray()

y_train = np.array(train_labels)
y_val = np.array(val_labels)

# **Step 3: Convert Data to PyTorch Tensors**
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)  # Binary classification

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

# **Step 4: Create DataLoader**
batch_size = 128
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# **Step 5: Define MLP Model for Scoring Choices**
class ChoiceScoringMLP(nn.Module):
    def __init__(self, input_dim):
        super(ChoiceScoringMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 1)  # Output a single score for the choice
        )

    def forward(self, x):
        return self.model(x).squeeze(1)  # Ensure output shape is (batch,)

# **Step 6: Initialize Model**
input_dim = X_train.shape[1]  # TF-IDF feature size
model_mlp = ChoiceScoringMLP(input_dim).to(device)

# **Step 7: Define Loss, Optimizer, and Scheduler**
criterion = nn.BCEWithLogitsLoss()  # Binary classification loss
optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=4, gamma=0.5)

# **Step 8: Train MLP with DataLoader & Scheduler**
num_epochs = 40
for epoch in range(num_epochs):
    model_mlp.train()
    epoch_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        outputs = model_mlp(batch_X)  # Get scores
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss / len(train_loader):.4f}, LR: {current_lr:.6f}")

# **Step 9: Evaluate Model**
model_mlp.eval()
y_pred_scores = []

with torch.no_grad():
    for batch_X, _ in val_loader:
        batch_X = batch_X.to(device)
        batch_scores = model_mlp(batch_X).cpu().numpy()
        y_pred_scores.extend(batch_scores)

# **Step 10: Compute Accuracy (Choice Selection)**
val_choice_indices = []  # Stores predicted choice index for each question
val_correct_indices = []  # Stores ground-truth indices

i = 0
for idx, row in val_df.iterrows():
    num_choices = len(row["choices"])
    scores = y_pred_scores[i:i + num_choices]  # Get scores for this question's choices
    pred_choice_idx = np.argmax(scores)  # Pick highest-scoring choice
    val_choice_indices.append(pred_choice_idx)
    val_correct_indices.append(row["correct_choice_idx"])  # Ground-truth index
    i += num_choices

# **Final Accuracy Computation**
bert_accuracy = accuracy_score(val_correct_indices, val_choice_indices)
print(f"TF-IDF + Choice-Based MLP Accuracy: {bert_accuracy:.2%}")

# **Step 11: Print Sample Predictions**
print("\nSample Predictions:")
i = 0
for idx, row in val_df.iterrows():
    if i >= 5:
        break
    print(f"Q: {row['question']}")
    print(f"Choices: {row['choices']}")
    print(f"Actual Answer: {row['choices'][row['correct_choice_idx']]}")
    print(f"Predicted Answer: {row['choices'][val_choice_indices[i]]}\n")
    i += 1

# Convert scores to probabilities using Sigmoid since BCEWithLogitsLoss was used
y_pred_probs = torch.sigmoid(torch.tensor(y_pred_scores)).numpy()
tak
# **Cross-Entropy Loss (Log Loss)**
cross_entropy = log_loss(y_val, y_pred_probs)
print(f"Cross-Entropy Loss: {cross_entropy:.4f}")

# **Precision, Recall, and F1 Score**
# Convert probabilities to binary predictions using a 0.5 threshold
y_pred_binary = (y_pred_probs >= 0.5).astype(int)

precision = precision_score(y_val, y_pred_binary)
recall = recall_score(y_val, y_pred_binary)
f1 = f1_score(y_val, y_pred_binary)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")



Epoch [1/40], Loss: 0.5866, LR: 0.001000
Epoch [2/40], Loss: 0.5741, LR: 0.001000
Epoch [3/40], Loss: 0.5704, LR: 0.001000
Epoch [4/40], Loss: 0.5678, LR: 0.000500
Epoch [5/40], Loss: 0.5610, LR: 0.000500
Epoch [6/40], Loss: 0.5589, LR: 0.000500
Epoch [7/40], Loss: 0.5563, LR: 0.000500
Epoch [8/40], Loss: 0.5527, LR: 0.000250
Epoch [9/40], Loss: 0.5300, LR: 0.000250
Epoch [10/40], Loss: 0.5109, LR: 0.000250
Epoch [11/40], Loss: 0.4800, LR: 0.000250
Epoch [12/40], Loss: 0.4390, LR: 0.000125
Epoch [13/40], Loss: 0.3565, LR: 0.000125
Epoch [14/40], Loss: 0.3014, LR: 0.000125
Epoch [15/40], Loss: 0.2613, LR: 0.000125
Epoch [16/40], Loss: 0.2291, LR: 0.000063
Epoch [17/40], Loss: 0.1788, LR: 0.000063
Epoch [18/40], Loss: 0.1526, LR: 0.000063
Epoch [19/40], Loss: 0.1376, LR: 0.000063
Epoch [20/40], Loss: 0.1286, LR: 0.000031
Epoch [21/40], Loss: 0.1052, LR: 0.000031
Epoch [22/40], Loss: 0.0968, LR: 0.000031
Epoch [23/40], Loss: 0.0918, LR: 0.000031
Epoch [24/40], Loss: 0.0872, LR: 0.000016
E

In [ ]:
y_val, y_pred_probs

(array([0, 0, 0, ..., 0, 0, 0]),
 array([0.36977327, 0.36642324, 0.34633724, ..., 0.26996935, 0.26241127,
        0.24049053]))

In [ ]:
val_pairs[:4], val_labels[:4]
for i in range(4):
  print(f"Question: {val_pairs[i][:20]}..., Choice: {val_pairs[i].split('.')[-1]}\nCorrect?: {val_labels[i]}")
print(f'Correct Idx for answer: {val_correct_indices[0]}\nOur Prediction: {val_choice_indices[0]}')

Question: What is in the motor..., Choice:  toothpick
Correct?: 0
Question: What is in the motor..., Choice:  food
Correct?: 0
Question: What is in the motor..., Choice:  popsicle stick
Correct?: 0
Question: What is in the motor..., Choice:  cigarette
Correct?: 1
Correct Idx for answer: 3
Our Prediction: 0


## TFIDF + MLP (Question ONLY)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import StepLR
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
import numpy as np

# **Step 1: Prepare Data - Create Pairs (Question + Choice)**
train_pairs = []
train_labels = []

val_pairs = []
val_labels = []

for i, row in train_df.iterrows():
    question = row["question"]
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]  # This is the correct choice index

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice  # Combine Question + Choice
        train_pairs.append(text_input)
        train_labels.append(1 if idx == correct_idx else 0)  # Label: 1 if correct, 0 otherwise

for i, row in val_df.iterrows():
    question = row["question"]
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice
        val_pairs.append(text_input)
        val_labels.append(1 if idx == correct_idx else 0)

# **Step 2: Compute TF-IDF Embeddings**
vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(train_pairs).toarray()
X_val = vectorizer.transform(val_pairs).toarray()

y_train = np.array(train_labels)
y_val = np.array(val_labels)

# **Step 3: Convert Data to PyTorch Tensors**
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)  # Binary classification

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

# **Step 4: Create DataLoader**
batch_size = 128
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# **Step 5: Define MLP Model for Scoring Choices**
class ChoiceScoringMLP(nn.Module):
    def __init__(self, input_dim):
        super(ChoiceScoringMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 1)  # Output a single score for the choice
        )

    def forward(self, x):
        return self.model(x).squeeze(1)  # Ensure output shape is (batch,)

# **Step 6: Initialize Model**
input_dim = X_train.shape[1]  # TF-IDF feature size
model_mlp = ChoiceScoringMLP(input_dim).to(device)

# **Step 7: Define Loss, Optimizer, and Scheduler**
criterion = nn.BCEWithLogitsLoss()  # Binary classification loss
optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=4, gamma=0.5)

# **Step 8: Train MLP with DataLoader & Scheduler**
num_epochs = 40
for epoch in range(num_epochs):
    model_mlp.train()
    epoch_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        outputs = model_mlp(batch_X)  # Get scores
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss / len(train_loader):.4f}, LR: {current_lr:.6f}")

# **Step 9: Evaluate Model**
model_mlp.eval()
y_pred_scores = []

with torch.no_grad():
    for batch_X, _ in val_loader:
        batch_X = batch_X.to(device)
        batch_scores = model_mlp(batch_X).cpu().numpy()
        y_pred_scores.extend(batch_scores)

# **Step 10: Compute Accuracy (Choice Selection)**
val_choice_indices = []  # Stores predicted choice index for each question
val_correct_indices = []  # Stores ground-truth indices

i = 0
for idx, row in val_df.iterrows():
    num_choices = len(row["choices"])
    scores = y_pred_scores[i:i + num_choices]  # Get scores for this question's choices
    pred_choice_idx = np.argmax(scores)  # Pick highest-scoring choice
    val_choice_indices.append(pred_choice_idx)
    val_correct_indices.append(row["correct_choice_idx"])  # Ground-truth index
    i += num_choices

# **Final Accuracy Computation**
bert_accuracy = accuracy_score(val_correct_indices, val_choice_indices)
print(f"TF-IDF + Choice-Based MLP Accuracy: {bert_accuracy:.2%}")

# **Step 11: Print Sample Predictions**
print("\nSample Predictions:")
i = 0
for idx, row in val_df.iterrows():
    if i >= 5:
        break
    print(f"Q: {row['question']}")
    print(f"Choices: {row['choices']}")
    print(f"Actual Answer: {row['choices'][row['correct_choice_idx']]}")
    print(f"Predicted Answer: {row['choices'][val_choice_indices[i]]}\n")
    i += 1


from sklearn.metrics import precision_score, recall_score, f1_score, log_loss

# Convert scores to probabilities using Sigmoid since BCEWithLogitsLoss was used
y_pred_probs = torch.sigmoid(torch.tensor(y_pred_scores)).numpy()

# **Cross-Entropy Loss (Log Loss)**
cross_entropy = log_loss(y_val, y_pred_probs)
print(f"Cross-Entropy Loss: {cross_entropy:.4f}")

# **Precision, Recall, and F1 Score**
# Convert probabilities to binary predictions using a 0.5 threshold
y_pred_binary = (y_pred_probs >= 0.5).astype(int)

precision = precision_score(y_val, y_pred_binary)
recall = recall_score(y_val, y_pred_binary)
f1 = f1_score(y_val, y_pred_binary)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")


Epoch [1/40], Loss: 0.5798, LR: 0.001000
Epoch [2/40], Loss: 0.5405, LR: 0.001000
Epoch [3/40], Loss: 0.5012, LR: 0.001000
Epoch [4/40], Loss: 0.4527, LR: 0.000500
Epoch [5/40], Loss: 0.3400, LR: 0.000500
Epoch [6/40], Loss: 0.2499, LR: 0.000500
Epoch [7/40], Loss: 0.1785, LR: 0.000500
Epoch [8/40], Loss: 0.1303, LR: 0.000250
Epoch [9/40], Loss: 0.0771, LR: 0.000250
Epoch [10/40], Loss: 0.0516, LR: 0.000250
Epoch [11/40], Loss: 0.0455, LR: 0.000250
Epoch [12/40], Loss: 0.0414, LR: 0.000125
Epoch [13/40], Loss: 0.0319, LR: 0.000125
Epoch [14/40], Loss: 0.0278, LR: 0.000125
Epoch [15/40], Loss: 0.0275, LR: 0.000125
Epoch [16/40], Loss: 0.0266, LR: 0.000063
Epoch [17/40], Loss: 0.0230, LR: 0.000063
Epoch [18/40], Loss: 0.0224, LR: 0.000063
Epoch [19/40], Loss: 0.0223, LR: 0.000063
Epoch [20/40], Loss: 0.0220, LR: 0.000031
Epoch [21/40], Loss: 0.0204, LR: 0.000031
Epoch [22/40], Loss: 0.0200, LR: 0.000031
Epoch [23/40], Loss: 0.0200, LR: 0.000031
Epoch [24/40], Loss: 0.0198, LR: 0.000016
E

## BERT + MLP (Question + Rationale)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gc
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import StepLR
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# **Load Pre-trained BERT Model and Tokenizer**
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased").to(device)

# **Function to Extract BERT Embeddings in Batches**
def get_bert_embeddings_batched(texts, batch_size=512, max_length=10000):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Processing Batches"):
        batch_texts = texts[i:i+batch_size]

        # Tokenize with truncation and padding
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)

        # Extract embeddings without gradient calculation
        with torch.no_grad():
            outputs = bert_model(**inputs)

        # Use CLS token embedding (first token)
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)

        # Clear memory
        del inputs, outputs
        torch.cuda.empty_cache()
        gc.collect()

    return np.vstack(all_embeddings)

# **Step 1: Prepare Data - Create Pairs (Question + Choice)**
train_pairs = []
train_labels = []

val_pairs = []
val_labels = []

for _, row in train_df.iterrows():
    question = row["question"] + row["rationale"]
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice  # Combine Question + Choice
        train_pairs.append(text_input)
        train_labels.append(1 if idx == correct_idx else 0)  # Label: 1 if correct, 0 otherwise

for _, row in val_df.iterrows():
    question = row["question"] + row["rationale"]
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice
        val_pairs.append(text_input)
        val_labels.append(1 if idx == correct_idx else 0)

# **Step 2: Compute BERT Embeddings**
print("Extracting BERT embeddings for training data...")
X_train = get_bert_embeddings_batched(train_pairs, batch_size=512, max_length=10000)
print("Extracting BERT embeddings for validation data...")
X_val = get_bert_embeddings_batched(val_pairs, batch_size=512, max_length=10000)

y_train = np.array(train_labels)
y_val = np.array(val_labels)

# **Step 3: Convert Data to PyTorch Tensors**
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)  # Binary classification

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

# **Step 4: Create DataLoader**
batch_size = 128
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# **Step 5: Define MLP Model for Scoring Choices**
class ChoiceScoringMLP(nn.Module):
    def __init__(self, input_dim):
        super(ChoiceScoringMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 1)  # Output a single score for the choice
        )

    def forward(self, x):
        return self.model(x).squeeze(1)  # Ensure output shape is (batch,)

# **Step 6: Initialize Model**
input_dim = X_train.shape[1]  # BERT embedding size (768 for BERT base)
model_mlp = ChoiceScoringMLP(input_dim).to(device)

# **Step 7: Define Loss, Optimizer, and Scheduler**
criterion = nn.BCEWithLogitsLoss()  # Binary classification loss
optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=4, gamma=0.5)

# **Step 8: Train MLP with DataLoader & Scheduler**
num_epochs = 40
for epoch in range(num_epochs):
    model_mlp.train()
    epoch_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        outputs = model_mlp(batch_X)  # Get scores
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss / len(train_loader):.4f}, LR: {current_lr:.6f}")

# **Step 9: Evaluate Model**
model_mlp.eval()
y_pred_scores = []

with torch.no_grad():
    for batch_X, _ in val_loader:
        batch_X = batch_X.to(device)
        batch_scores = model_mlp(batch_X).cpu().numpy()
        y_pred_scores.extend(batch_scores)

# **Step 10: Compute Accuracy (Choice Selection)**
val_choice_indices = []  # Stores predicted choice index for each question
val_correct_indices = []  # Stores ground-truth indices

i = 0
for _, row in val_df.iterrows():
    num_choices = len(row["choices"])
    scores = y_pred_scores[i:i + num_choices]  # Get scores for this question's choices
    pred_choice_idx = np.argmax(scores)  # Pick highest-scoring choice
    val_choice_indices.append(pred_choice_idx)
    val_correct_indices.append(row["correct_choice_idx"])  # Ground-truth index
    i += num_choices

# **Final Accuracy Computation**
bert_accuracy = accuracy_score(val_correct_indices, val_choice_indices)
print(f"BERT + Choice-Based MLP Accuracy: {bert_accuracy:.2%}")

# **Step 11: Print Sample Predictions**
print("\nSample Predictions:")
i = 0
for _, row in val_df.iterrows():
    if i >= 5:
        break
    print(f"Q: {row['question']}")
    print(f"Choices: {row['choices']}")
    print(f"Actual Answer: {row['choices'][row['correct_choice_idx']]}")
    print(f"Predicted Answer: {row['choices'][val_choice_indices[i]]}\n")
    i += 1


# Convert scores to probabilities using Sigmoid since BCEWithLogitsLoss was used
y_pred_probs = torch.sigmoid(torch.tensor(y_pred_scores)).numpy()

# **Cross-Entropy Loss (Log Loss)**
cross_entropy = log_loss(y_val, y_pred_probs)
print(f"Cross-Entropy Loss: {cross_entropy:.4f}")

# **Precision, Recall, and F1 Score**
# Convert probabilities to binary predictions using a 0.5 threshold
y_pred_binary = (y_pred_probs >= 0.5).astype(int)

precision = precision_score(y_val, y_pred_binary)
recall = recall_score(y_val, y_pred_binary)
f1 = f1_score(y_val, y_pred_binary)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")



## BERT + MLP (Question ONLY)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gc
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import StepLR
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# **Load Pre-trained BERT Model and Tokenizer**
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased").to(device)

# **Function to Extract BERT Embeddings in Batches**
def get_bert_embeddings_batched(texts, batch_size=512, max_length=10000):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Processing Batches"):
        batch_texts = texts[i:i+batch_size]

        # Tokenize with truncation and padding
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)

        # Extract embeddings without gradient calculation
        with torch.no_grad():
            outputs = bert_model(**inputs)

        # Use CLS token embedding (first token)
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)

        # Clear memory
        del inputs, outputs
        torch.cuda.empty_cache()
        gc.collect()

    return np.vstack(all_embeddings)

# **Step 1: Prepare Data - Create Pairs (Question + Choice)**
train_pairs = []
train_labels = []

val_pairs = []
val_labels = []

for _, row in train_df.iterrows():
    question = row["question"]
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice  # Combine Question + Choice
        train_pairs.append(text_input)
        train_labels.append(1 if idx == correct_idx else 0)  # Label: 1 if correct, 0 otherwise

for _, row in val_df.iterrows():
    question = row["question"]
    choices = row["choices"]
    correct_idx = row["correct_choice_idx"]

    for idx, choice in enumerate(choices):
        text_input = question + " " + choice
        val_pairs.append(text_input)
        val_labels.append(1 if idx == correct_idx else 0)

# **Step 2: Compute BERT Embeddings**
print("Extracting BERT embeddings for training data...")
X_train = get_bert_embeddings_batched(train_pairs, batch_size=512, max_length=10000)
print("Extracting BERT embeddings for validation data...")
X_val = get_bert_embeddings_batched(val_pairs, batch_size=512, max_length=10000)

y_train = np.array(train_labels)
y_val = np.array(val_labels)

# **Step 3: Convert Data to PyTorch Tensors**
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)  # Binary classification

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

# **Step 4: Create DataLoader**
batch_size = 128
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# **Step 5: Define MLP Model for Scoring Choices**
class ChoiceScoringMLP(nn.Module):
    def __init__(self, input_dim):
        super(ChoiceScoringMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 1)  # Output a single score for the choice
        )

    def forward(self, x):
        return self.model(x).squeeze(1)  # Ensure output shape is (batch,)

# **Step 6: Initialize Model**
input_dim = X_train.shape[1]  # BERT embedding size (768 for BERT base)
model_mlp = ChoiceScoringMLP(input_dim).to(device)

# **Step 7: Define Loss, Optimizer, and Scheduler**
criterion = nn.BCEWithLogitsLoss()  # Binary classification loss
optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=4, gamma=0.5)

# **Step 8: Train MLP with DataLoader & Scheduler**
num_epochs = 40
for epoch in range(num_epochs):
    model_mlp.train()
    epoch_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        outputs = model_mlp(batch_X)  # Get scores
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss / len(train_loader):.4f}, LR: {current_lr:.6f}")

# **Step 9: Evaluate Model**
model_mlp.eval()
y_pred_scores = []

with torch.no_grad():
    for batch_X, _ in val_loader:
        batch_X = batch_X.to(device)
        batch_scores = model_mlp(batch_X).cpu().numpy()
        y_pred_scores.extend(batch_scores)

# **Step 10: Compute Accuracy (Choice Selection)**
val_choice_indices = []  # Stores predicted choice index for each question
val_correct_indices = []  # Stores ground-truth indices

i = 0
for _, row in val_df.iterrows():
    num_choices = len(row["choices"])
    scores = y_pred_scores[i:i + num_choices]  # Get scores for this question's choices
    pred_choice_idx = np.argmax(scores)  # Pick highest-scoring choice
    val_choice_indices.append(pred_choice_idx)
    val_correct_indices.append(row["correct_choice_idx"])  # Ground-truth index
    i += num_choices

# **Final Accuracy Computation**
bert_accuracy = accuracy_score(val_correct_indices, val_choice_indices)
print(f"BERT + Choice-Based MLP Accuracy: {bert_accuracy:.2%}")

# **Step 11: Print Sample Predictions**
print("\nSample Predictions:")
i = 0
for _, row in val_df.iterrows():
    if i >= 5:
        break
    print(f"Q: {row['question']}")
    print(f"Choices: {row['choices']}")
    print(f"Actual Answer: {row['choices'][row['correct_choice_idx']]}")
    print(f"Predicted Answer: {row['choices'][val_choice_indices[i]]}\n")
    i += 1
# Convert scores to probabilities using Sigmoid since BCEWithLogitsLoss was used
y_pred_probs = torch.sigmoid(torch.tensor(y_pred_scores)).numpy()

# **Cross-Entropy Loss (Log Loss)**
cross_entropy = log_loss(y_val, y_pred_probs)
print(f"Cross-Entropy Loss: {cross_entropy:.4f}")

# **Precision, Recall, and F1 Score**
# Convert probabilities to binary predictions using a 0.5 threshold
y_pred_binary = (y_pred_probs >= 0.5).astype(int)

precision = precision_score(y_val, y_pred_binary)
recall = recall_score(y_val, y_pred_binary)
f1 = f1_score(y_val, y_pred_binary)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")



Extracting BERT embeddings for training data...


Processing Batches: 100%|██████████| 134/134 [01:18<00:00,  1.71it/s]


Extracting BERT embeddings for validation data...


Processing Batches: 100%|██████████| 9/9 [00:05<00:00,  1.71it/s]


Epoch [1/40], Loss: 0.5660, LR: 0.001000
Epoch [2/40], Loss: 0.5499, LR: 0.001000
Epoch [3/40], Loss: 0.5405, LR: 0.001000
Epoch [4/40], Loss: 0.5308, LR: 0.000500
Epoch [5/40], Loss: 0.5053, LR: 0.000500
Epoch [6/40], Loss: 0.4875, LR: 0.000500
Epoch [7/40], Loss: 0.4658, LR: 0.000500
Epoch [8/40], Loss: 0.4401, LR: 0.000250
Epoch [9/40], Loss: 0.3708, LR: 0.000250
Epoch [10/40], Loss: 0.3285, LR: 0.000250
Epoch [11/40], Loss: 0.2880, LR: 0.000250
Epoch [12/40], Loss: 0.2518, LR: 0.000125
Epoch [13/40], Loss: 0.1880, LR: 0.000125
Epoch [14/40], Loss: 0.1601, LR: 0.000125
Epoch [15/40], Loss: 0.1427, LR: 0.000125
Epoch [16/40], Loss: 0.1258, LR: 0.000063
Epoch [17/40], Loss: 0.0970, LR: 0.000063
Epoch [18/40], Loss: 0.0873, LR: 0.000063
Epoch [19/40], Loss: 0.0803, LR: 0.000063
Epoch [20/40], Loss: 0.0738, LR: 0.000031
Epoch [21/40], Loss: 0.0643, LR: 0.000031
Epoch [22/40], Loss: 0.0590, LR: 0.000031
Epoch [23/40], Loss: 0.0568, LR: 0.000031
Epoch [24/40], Loss: 0.0547, LR: 0.000016
E